# **Tugas 1 – Web Scraping Berita Detik.com untuk Pengumpulan Dataset Kategori Sport dan Finance**
---


## **Instalasi Library**

---

Pada tahap pertama dilakukan instalasi beberapa library Python yang dibutuhkan untuk proses Web Scraping. Library yang digunakan yaitu `requests` untuk mengambil halaman web, `beautifulsoup4` untuk membaca struktur HTML, `trafilatura` untuk mengambil isi utama artikel, `pandas` untuk mengolah data, `openpyxl` untuk menyimpan data ke dalam file Excel, dan `tqdm` untuk menampilkan progress proses scraping.

In [ ]:
# Jika belum install library, jalankan sekali saja
!pip install requests beautifulsoup4 trafilatura pandas openpyxl tqdm


   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [et-xmlfile]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
  

## **Import Library dan Konfigurasi**

---



Pada tahap ini dilakukan import library yang akan digunakan dalam proses Web Scraping. Library `re` digunakan untuk melakukan manipulasi teks, `time` dan `random` digunakan untuk memberikan jeda secara acak selama proses scraping, sedangkan `requests` digunakan untuk mengambil halaman web. Library `pandas` digunakan untuk mengolah data dan `trafilatura` digunakan untuk mengambil isi utama artikel. Selain itu, `BeautifulSoup` digunakan untuk membaca struktur HTML, sedangkan beberapa fungsi dari `urllib.parse` digunakan untuk mengolah URL. Library `tqdm` digunakan untuk menampilkan progress proses scraping. Pada bagian konfigurasi juga dibuat `User-Agent` dan `requests.Session()` yang digunakan untuk mengatur request ke halaman web. Variabel `kategori` digunakan untuk menyimpan alamat halaman indeks dan domain berita yang akan digunakan, yaitu kategori sport dan finance.
---

In [ ]:
import re
import time
import random
import requests
import pandas as pd
import trafilatura

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit, urlunsplit
from tqdm import tqdm


# ============================================================
# 1. KONFIGURASI
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


kategori = {
    "sport": {
        "url": "https://sport.detik.com/indeks",
        "domain": "sport.detik.com"
    },

    "finance": {
        "url": "https://finance.detik.com/indeks",
        "domain": "finance.detik.com"
    }
}

## **Normalisasi URL**

---



Fungsi `normalize_url()` digunakan untuk membersihkan URL yang diperoleh dari halaman web. Pada proses ini query parameter dan fragment pada URL dihapus sehingga URL menjadi lebih sederhana dan konsisten. Fungsi `urlsplit()` digunakan untuk memisahkan bagian-bagian URL, sedangkan `urlunsplit()` digunakan untuk menyusun kembali URL yang telah dibersihkan. Normalisasi ini juga membantu menghindari adanya link artikel yang sama tetapi memiliki URL yang berbeda karena adanya parameter tambahan.
---

In [ ]:
def normalize_url(url):
    """
    Menghapus query parameter dan fragment dari URL.
    """

    bagian = urlsplit(url)

    url_bersih = urlunsplit(
        (
            bagian.scheme,
            bagian.netloc,
            bagian.path,
            "",
            ""
        )
    )

    return url_bersih

## **Mengambil Link Berita**

---



Fungsi `ambil_link_berita()` digunakan untuk mengambil link artikel dari halaman indeks Detik berdasarkan kategori berita. Program membaca halaman indeks secara bertahap menggunakan nomor halaman, kemudian memeriksa link yang ditemukan untuk memastikan bahwa link tersebut berasal dari domain yang sesuai dan memiliki pola URL artikel Detik. Link yang sudah ditemukan disimpan menggunakan `set` untuk menghindari duplikasi. Parameter `target` digunakan untuk menentukan jumlah link yang ingin dikumpulkan, sedangkan `max_page` digunakan untuk membatasi jumlah halaman yang diperiksa. Program juga memberikan jeda secara acak antara satu sampai dua detik pada setiap halaman agar request tidak dilakukan secara terus-menerus tanpa jeda.
---

In [ ]:
def ambil_link_berita(index_url, domain, target=150, max_page=20):

    links = []
    sudah_ada = set()

    for page in range(1, max_page + 1):

        url_page = f"{index_url}?page={page}"

        print(f"Membaca halaman {page}")

        try:

            response = session.get(
                url_page,
                timeout=20
            )

            response.raise_for_status()

        except Exception as e:

            print("Error:", e)
            continue


        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        for tag in soup.find_all("a", href=True):

            href = urljoin(
                url_page,
                tag["href"]
            )

            href = normalize_url(href)

            parsed = urlsplit(href)


            # URL artikel Detik biasanya memiliki /d-xxxx
            if (
                parsed.netloc == domain
                and re.search(r"/d-\d+", parsed.path)
            ):

                if href not in sudah_ada:

                    sudah_ada.add(href)

                    links.append(href)


        print(
            "Jumlah link:",
            len(links)
        )


        if len(links) >= target:
            break


        # delay
        time.sleep(
            random.uniform(1, 2)
        )


    return links

## **Pengumpulan Link Berita Sport**

---



Pada tahap ini dilakukan pengumpulan kandidat link berita dari kategori `sport` menggunakan fungsi `ambil_link_berita()`. Target awal yang ditentukan adalah sebanyak 150 link berita. Program akan membaca beberapa halaman indeks dan mengumpulkan link artikel yang memenuhi kriteria. Hasil link yang diperoleh kemudian disimpan pada variabel `link_sport`. Berdasarkan hasil eksekusi, diperoleh sebanyak **157 kandidat link berita sport** yang selanjutnya digunakan sebagai sumber untuk mengambil isi artikel.
---

In [ ]:
link_sport = ambil_link_berita(
    kategori["sport"]["url"],
    kategori["sport"]["domain"],
    target=150
)

print(
    "Total kandidat Sport:",
    len(link_sport)
)

Membaca halaman 1
Jumlah link: 20
Membaca halaman 2
Jumlah link: 40
Membaca halaman 3
Jumlah link: 60
Membaca halaman 4
Jumlah link: 79
Membaca halaman 5
Jumlah link: 98
Membaca halaman 6
Jumlah link: 117
Membaca halaman 7
Jumlah link: 137
Membaca halaman 8
Jumlah link: 157
Total kandidat Sport: 157


## **Pengumpulan Kandidat Link Berita Finance**

---



Pada tahap ini dilakukan pengumpulan kandidat link berita dari kategori `finance` menggunakan fungsi `ambil_link_berita()`. Target yang ditentukan adalah sebanyak 150 link berita. Program akan membaca halaman indeks dan mengambil link artikel yang sesuai dengan domain serta pola URL yang telah ditentukan. Hasil pengumpulan link kemudian disimpan pada variabel `link_finance`. Berdasarkan hasil eksekusi, diperoleh sebanyak **154 kandidat link berita finance** yang akan digunakan pada proses scraping isi artikel.
---

In [ ]:
link_finance = ambil_link_berita(
    kategori["finance"]["url"],
    kategori["finance"]["domain"],
    target=150
)

print(
    "Total kandidat Finance:",
    len(link_finance)
)

Membaca halaman 1
Jumlah link: 18
Membaca halaman 2
Jumlah link: 38
Membaca halaman 3
Jumlah link: 57
Membaca halaman 4
Jumlah link: 77
Membaca halaman 5
Jumlah link: 94
Membaca halaman 6
Jumlah link: 114
Membaca halaman 7
Jumlah link: 134
Membaca halaman 8
Jumlah link: 154
Total kandidat Finance: 154


## **Pembersihan Teks Berita**

---



Fungsi `bersihkan_teks()` digunakan untuk membersihkan isi berita yang diperoleh dari halaman web. Beberapa teks yang tidak termasuk isi utama berita, seperti `ADVERTISEMENT` dan `SCROLL TO CONTINUE WITH CONTENT`, dihapus menggunakan regular expression. Selain itu, spasi atau whitespace yang berlebihan diubah menjadi satu spasi agar teks menjadi lebih rapi. Jika teks yang diperoleh bernilai `None`, fungsi akan mengembalikan `None`. Setelah seluruh proses selesai, teks dirapikan menggunakan `strip()` untuk menghilangkan spasi pada bagian awal dan akhir.
---

In [ ]:
def bersihkan_teks(teks):

    if teks is None:
        return None


    # menghapus teks iklan
    teks = re.sub(
        r"ADVERTISEMENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    teks = re.sub(
        r"SCROLL TO CONTINUE WITH CONTENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    # menghapus spasi berlebihan
    teks = re.sub(
        r"\s+",
        " ",
        teks
    )


    return teks.strip()

## **Ekstraksi Isi Berita**

---



Fungsi `ambil_isi_berita()` digunakan untuk mengambil isi utama artikel berdasarkan URL yang telah dikumpulkan. Program melakukan request ke halaman artikel menggunakan `session.get()`, kemudian memeriksa keberhasilan request menggunakan `raise_for_status()`. Setelah halaman berhasil diperoleh, `trafilatura.extract()` digunakan untuk mengekstrak teks utama artikel dengan tidak menyertakan komentar dan tabel. Hasil ekstraksi kemudian dibersihkan menggunakan fungsi `bersihkan_teks()`. Artikel hanya diterima apabila memiliki isi dan panjang teks minimal 300 karakter. Jika terjadi kesalahan saat proses pengambilan artikel, program menampilkan URL dan pesan error kemudian mengembalikan nilai `None`.
---

In [ ]:
def ambil_isi_berita(url):

    try:

        response = session.get(
            url,
            timeout=25
        )

        response.raise_for_status()


        isi = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False,
            output_format="txt",
            favor_precision=True,
            deduplicate=True
        )


        isi = bersihkan_teks(isi)


        # menghindari halaman kosong
        # atau isi yang terlalu pendek
        if isi and len(isi) >= 300:

            return isi


    except Exception as e:

        print(
            "Gagal:",
            url
        )

        print(
            "Error:",
            e
        )


    return None

## **Proses Scraping Berdasarkan Kategori**

---



Fungsi `scraping_kategori()` digunakan untuk mengambil isi berita dari kumpulan link berdasarkan kategori tertentu. Setiap URL diproses menggunakan fungsi `ambil_isi_berita()`. Jika isi artikel berhasil diperoleh, data disimpan dalam bentuk dictionary yang terdiri dari `isi_berita` dan `label`. Parameter `target` digunakan untuk menentukan jumlah artikel yang ingin dikumpulkan. Progress proses ditampilkan menggunakan `tqdm`, sedangkan delay acak diberikan setelah setiap proses pengambilan artikel agar request tidak dilakukan terlalu cepat.
---

In [ ]:
def scraping_kategori(
    links,
    label,
    target=100
):

    hasil = []


    for url in tqdm(
        links,
        desc=f"Scraping {label}"
    ):


        if len(hasil) >= target:
            break


        isi = ambil_isi_berita(url)


        if isi is not None:

            hasil.append(
                {
                    "isi_berita": isi,
                    "label": label
                }
            )


        # delay supaya tidak terlalu cepat
        time.sleep(
            random.uniform(1, 2)
        )


    print(
        f"{label} berhasil:",
        len(hasil)
    )


    return hasil

## **Scraping 100 Berita Sport**

---



Pada tahap ini dilakukan proses scraping isi artikel dari kandidat link kategori `sport` dan `finance`. Fungsi `scraping_kategori()` digunakan dengan target sebanyak 100 artikel. Program akan mengambil isi berita dari link yang tersedia dan hanya memasukkan artikel yang berhasil diekstraksi sesuai kriteria yang telah ditentukan. Hasil proses kemudian disimpan dalam variabel `data_sport` dan `data_finance`. Berdasarkan hasil eksekusi, berhasil diperoleh **100 berita sport** dan **100 berita finance**.
---

In [ ]:
data_sport = scraping_kategori(
    link_sport,
    "sport",
    target=100
)

Scraping sport:  66%|██████▌   | 103/157 [04:20<02:16,  2.53s/it]

sport berhasil: 100


In [ ]:
data_finance = scraping_kategori(
    link_finance,
    "finance",
    target=100
)

Scraping finance:  68%|██████▊   | 105/154 [04:15<01:59,  2.44s/it]

finance berhasil: 100


## **Menggabungkan Data Berita**

---



Pada tahap ini data berita dari kategori `sport` dan `finance` digabungkan menjadi satu data menggunakan operator `+`. Data dari kedua kategori yang sebelumnya disimpan dalam variabel berbeda digabungkan ke dalam variabel `data_semua`. Dengan demikian, seluruh hasil scraping dari kedua kategori dapat diproses sebagai satu dataset.
---

In [ ]:
data_semua = (
    data_sport
    +
    data_finance
)

In [ ]:
df = pd.DataFrame(
    data_semua
)

In [ ]:
df.insert(
    0,
    "id",
    range(
        1,
        len(df) + 1
    )
)

In [ ]:
df = df[
    [
        "id",
        "isi_berita",
        "label"
    ]
]

In [ ]:
df

,id,isi_berita,label
0,1,Pacuan kuda Indonesia mulai diarahkan menuju p...,sport
1,2,"Jelang Asian Games 2026, dukungan untuk 432 at...",sport
2,3,Nova Widianto mulai menangani tim ganda campur...,sport
3,4,Timnas basket putra Indonesia jadi rombongan p...,sport
4,5,"Bagi pelari urban, lari kini bukan lagi sekada...",sport
...,...,...,...
195,196,"Menteri Pertanian (Mentan), Andi Amran Sulaima...",finance
196,197,Menteri Keuangan (Menkeu) Purbaya Yudhi mengaj...,finance
197,198,Masyarakat Indonesia semakin banyak yang mengg...,finance
198,199,Presiden Prabowo Subianto meminta semua pihak ...,finance


In [ ]:
print(
    "Total data:",
    len(df)
)

Total data: 200


In [ ]:
print(
    df["label"].value_counts()
)

label
sport      100
finance    100
Name: count, dtype: int64


In [ ]:
nama_file = "dataset_detik_200_berita.xlsx"


df.to_excel(
    nama_file,
    index=False
)


print(
    "File berhasil disimpan:",
    nama_file
)

File berhasil disimpan: dataset_detik_200_berita.xlsx


In [ ]:
import os

print(os.path.abspath("dataset_detik_200_berita.xlsx"))

C:\Users\Asus\ppw\dataset_detik_200_berita.xlsx
